<a href="https://colab.research.google.com/github/noyeem99/privacy-preserving-deep-learning/blob/main/13.%20randomness-dp-sgd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class ResearchModel(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=64, output_dim=2):
        super(ResearchModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class DifferentiallyPrivateCorrelatedOptimizer:
    def __init__(self, base_optimizer, model, noise_multiplier, max_grad_norm, lambda_corr):
        self.optimizer = base_optimizer
        self.model = model
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm
        self.lambda_corr = lambda_corr
        self.previous_noise = {}
        self.loss_fn = nn.CrossEntropyLoss(reduction='none')

    def step(self, data, targets):
        batch_size = data.size(0)
        self.optimizer.zero_grad()

        def compute_loss(params, buffers, single_x, single_y):
            out = torch.func.functional_call(self.model, (params, buffers), single_x.unsqueeze(0))
            loss = self.loss_fn(out, single_y.unsqueeze(0))
            return loss.sum()

        params = dict(self.model.named_parameters())
        buffers = dict(self.model.named_buffers())

        grad_fn = torch.func.grad(compute_loss, argnums=0)
        per_sample_grads_dict = torch.func.vmap(grad_fn, in_dims=(None, None, 0, 0))(params, buffers, data, targets)

        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if param.requires_grad is False:
                    continue

                per_sample_grads = per_sample_grads_dict[name]
                flat_grads = per_sample_grads.flatten(start_dim=1)
                per_sample_norms = torch.norm(flat_grads, p=2, dim=1)
                clip_factors = torch.clamp(self.max_grad_norm / (per_sample_norms + 1e-6), max=1.0)

                view_shape = [batch_size] + [1] * (per_sample_grads.dim() - 1)
                clipped_grads = per_sample_grads * clip_factors.view(view_shape)
                param.grad = torch.mean(clipped_grads, dim=0)

                noise_scale = (self.noise_multiplier * self.max_grad_norm) / batch_size
                raw_noise = torch.randn_like(param.grad) * noise_scale

                if name in self.previous_noise:
                    corr_noise = (self.lambda_corr * self.previous_noise[name]) + \
                                 (torch.sqrt(torch.tensor(1.0 - self.lambda_corr**2)) * raw_noise)
                else:
                    corr_noise = raw_noise

                self.previous_noise[name] = corr_noise
                param.grad.add_(corr_noise)

        self.optimizer.step()

    def zero_grad(self):
        self.optimizer.zero_grad()

if __name__ == "__main__":
    # 1. Initialize Model and Optimizer
    model = ResearchModel()
    base_opt = optim.AdamW(model.parameters(), lr=0.001)

    dp_optimizer = DifferentiallyPrivateCorrelatedOptimizer(
        base_optimizer=base_opt,
        model=model,
        noise_multiplier=1.2,
        max_grad_norm=1.0,
        lambda_corr=0.5
    )

    # 2. Generate 100 Sample Dummy Dataset
    x_train_dummy = torch.randn(100, 100)
    y_train_dummy = torch.randint(0, 2, (100,))

    dataset = TensorDataset(x_train_dummy, y_train_dummy)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

    # 3. Execution Verification Loop
    print("Starting training check loop...")
    for epoch in range(2):
        for step, (batch_x, batch_y) in enumerate(dataloader):
            dp_optimizer.step(batch_x, batch_y)
            if step % 5 == 0:
                print(f"Epoch {epoch+1} - Step {step}: Step completed successfully.")

    print("Optimization Step Executed Successfully with True Per-Sample Privacy!")


Starting training check loop...
Epoch 1 - Step 0: Step completed successfully.
Epoch 1 - Step 5: Step completed successfully.
Epoch 1 - Step 10: Step completed successfully.
Epoch 1 - Step 15: Step completed successfully.
Epoch 1 - Step 20: Step completed successfully.
Epoch 2 - Step 0: Step completed successfully.
Epoch 2 - Step 5: Step completed successfully.
Epoch 2 - Step 10: Step completed successfully.
Epoch 2 - Step 15: Step completed successfully.
Epoch 2 - Step 20: Step completed successfully.
Optimization Step Executed Successfully with True Per-Sample Privacy!
